In [1]:
!pip install opencv-python==4.11.0.86 matplotlib==3.9.4

In [2]:
import sys
sys.path.append(r'c:\ai_project01\Pytorch_Retinaface')

In [3]:
import torch
import cv2
import numpy as np

from models.retinaface import RetinaFace
from data import cfg_re50
from layers.functions.prior_box import PriorBox
from utils.nms.py_cpu_nms import py_cpu_nms
from utils.box_utils import decode

In [4]:
torch.cuda.is_available()

True

In [5]:
device = "cuda"

In [ ]:
cfg = cfg_re50
net = RetinaFace(cfg=cfg, phase='test').to(device)

c:\Users\user\anaconda3\envs\yolo_env01\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\anaconda3\envs\yolo_env01\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
pretrained_path = r'c:\ai_project01\Pytorch_Retinaface\weights\Resnet50_Final.pth'
state_dict = torch.load(pretrained_path, map_location=device)

C:\Users\user\AppData\Local\Temp\ipykernel_5236\1137602229.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_path, map_location=device)


In [8]:
state_dict

OrderedDict([('module.body.conv1.weight',
              tensor([[[[ 1.8582e-03,  1.0580e-02,  1.5584e-03,  ..., -1.3624e-02,
                         -1.7488e-02, -4.5827e-02],
                        [ 3.4468e-03,  1.1286e-02,  3.1250e-02,  ...,  2.2371e-02,
                          2.7780e-03, -6.0711e-03],
                        [ 1.3523e-02,  2.1148e-02,  1.7607e-02,  ...,  7.6721e-02,
                          4.1229e-02,  4.4042e-02],
                        ...,
                        [-2.3033e-02,  2.1630e-02, -1.0141e-02,  ..., -1.2473e-01,
                         -8.1671e-02, -1.2715e-02],
                        [-4.6688e-04,  4.7038e-02,  5.1219e-02,  ..., -3.6879e-03,
                         -4.0615e-02, -1.3927e-02],
                        [-2.7420e-02,  5.0066e-03,  1.3666e-02,  ...,  3.7602e-02,
                          3.4055e-02,  3.7086e-02]],
              
                       [[-6.9415e-03,  1.4870e-02,  3.0648e-02,  ...,  5.1590e-02,
                    

In [9]:
new_state_dic = {}
for k, v in state_dict.items():
    if k.startswith("module."):

        new_state_dic[k[7:]]=v
    else:
        print("이름이 module. 으로 시작 안해")
        new_state_dic[k]=v

In [10]:
net.load_state_dict(new_state_dic, strict=True)

<All keys matched successfully>

In [11]:
net.eval()

RetinaFace(
  (body): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Seque

In [12]:
image_path = r'C:\ai_project01\mask_images\no_mask\no_mask_00000.png'

img_raw = cv2.imread(image_path, cv2.IMREAD_COLOR)

img = np.float32(img_raw)

In [13]:
im_height, im_width, _ = img.shape

img -= (104,117,123)

img = img.transpose(2,0,1)

img = torch.from_numpy(img).unsqueeze(0).to("cuda")

In [14]:
loc, conf, landms = net(img)

In [15]:
priorbox = PriorBox(cfg, image_size=(im_height, im_width))

In [16]:
priors = priorbox.forward().to("cuda")

In [17]:
prior_data = priors.data

In [18]:
prior_data.shape

torch.Size([16800, 4])

In [19]:
loc.data.squeeze(0).shape

torch.Size([16800, 4])

In [68]:
boxes = decode(loc.data.squeeze(0), prior_data, cfg["variance"])

In [45]:
scale=torch.Tensor([im_width,im_height,im_width,im_height]).to("cuda")

In [69]:
boxes

tensor([[-1.3606e-03, -1.5960e-03,  1.7856e-02,  2.2200e-02],
        [-8.9754e-03, -1.0256e-02,  3.3185e-02,  4.9249e-02],
        [ 5.8482e-03, -4.0988e-04,  2.9118e-02,  2.1940e-02],
        ...,
        [ 5.6583e-01,  6.0727e-01,  1.2615e+00,  1.3013e+00],
        [ 8.2517e-01,  7.9800e-01,  1.1163e+00,  1.1424e+00],
        [ 7.1418e-01,  6.7272e-01,  1.2554e+00,  1.2769e+00]], device='cuda:0')

In [49]:
scale

tensor([640., 640., 640., 640.], device='cuda:0')

In [71]:
boxes = boxes * scale

In [51]:
scores = conf.squeeze(0).data.cpu().numpy()[:, 1]

In [52]:
scores

array([1.2796046e-04, 4.5725388e-05, 1.3636245e-04, ..., 2.1540077e-04,
       5.4207369e-05, 1.3465319e-05], dtype=float32)

In [53]:
confience_threshold = 0.5

In [54]:
scores > confience_threshold

array([False, False, False, ..., False, False, False])

In [55]:
top_indices=np.where(scores > confience_threshold)[0]

In [73]:
boxes = boxes[top_indices]

In [57]:
scores = scores[top_indices]

In [66]:
boxes.cpu().numpy()

array([[244.35141 ,  81.27611 , 396.2239  , 298.01505 ],
       [246.53369 ,  85.132225, 395.75702 , 295.89786 ],
       [244.86946 ,  85.66684 , 396.02606 , 295.4181  ],
       [245.68753 ,  83.911446, 398.28143 , 294.414   ],
       [245.39587 ,  83.11183 , 392.13974 , 294.49374 ],
       [247.1502  ,  85.258095, 395.9555  , 294.81265 ],
       [245.26294 ,  85.829346, 395.5988  , 293.8523  ],
       [244.70055 ,  85.54856 , 395.74203 , 295.45343 ],
       [244.04272 ,  87.54815 , 396.62115 , 293.86084 ],
       [244.33565 ,  87.060135, 399.15298 , 292.8581  ],
       [245.0539  ,  85.102234, 394.248   , 295.84143 ],
       [245.2376  ,  84.18934 , 395.93256 , 297.58304 ],
       [245.35446 ,  82.7551  , 396.4058  , 297.94864 ],
       [244.88208 ,  83.058945, 397.00803 , 297.48877 ],
       [243.99893 ,  84.73763 , 396.35242 , 296.33926 ],
       [244.29169 ,  89.284454, 397.68097 , 294.79968 ],
       [243.53824 ,  87.32167 , 395.1401  , 296.0339  ],
       [245.04845 ,  84.3929  ,

In [39]:
scores[:, np.newaxis]

array([[0.8229067 ],
       [0.9818268 ],
       [0.98265713],
       [0.8864893 ],
       [0.5517854 ],
       [0.98759973],
       [0.9984635 ],
       [0.9986992 ],
       [0.99331445],
       [0.64307535],
       [0.7486323 ],
       [0.9946437 ],
       [0.9982998 ],
       [0.99900407],
       [0.9970023 ],
       [0.8566816 ],
       [0.7270132 ],
       [0.99225664],
       [0.9981488 ],
       [0.999044  ],
       [0.9968527 ],
       [0.81772715],
       [0.64260113],
       [0.9907781 ],
       [0.9984438 ],
       [0.999044  ],
       [0.9965668 ],
       [0.7302477 ],
       [0.6232117 ],
       [0.9921268 ],
       [0.9983903 ],
       [0.9988166 ],
       [0.9964887 ],
       [0.741264  ],
       [0.98672026],
       [0.9976017 ],
       [0.9976954 ],
       [0.9908455 ],
       [0.5739488 ],
       [0.8423493 ],
       [0.98692095],
       [0.9828448 ],
       [0.86846393],
       [0.9439675 ],
       [0.94811577],
       [0.951072  ],
       [0.9423687 ],
       [0.742

In [40]:
dets = np.hstack((boxes.cpu().numpy(), scores[:, np.newaxis])).astype(np.float32, copy=False)

In [41]:
dets

array([[244.35141   ,  81.27611   , 396.2239    , 298.01505   ,
          0.8229067 ],
       [246.53369   ,  85.132225  , 395.75702   , 295.89786   ,
          0.9818268 ],
       [244.86946   ,  85.66684   , 396.02606   , 295.4181    ,
          0.98265713],
       [245.68753   ,  83.911446  , 398.28143   , 294.414     ,
          0.8864893 ],
       [245.39587   ,  83.11183   , 392.13974   , 294.49374   ,
          0.5517854 ],
       [247.1502    ,  85.258095  , 395.9555    , 294.81265   ,
          0.98759973],
       [245.26294   ,  85.829346  , 395.5988    , 293.8523    ,
          0.9984635 ],
       [244.70055   ,  85.54856   , 395.74203   , 295.45343   ,
          0.9986992 ],
       [244.04272   ,  87.54815   , 396.62115   , 293.86084   ,
          0.99331445],
       [244.33565   ,  87.060135  , 399.15298   , 292.8581    ,
          0.64307535],
       [245.0539    ,  85.102234  , 394.248     , 295.84143   ,
          0.7486323 ],
       [245.2376    ,  84.18934   , 395.932

In [42]:
keep = py_cpu_nms(dets, 0.3)

In [43]:
dets[keep, : ]

array([[243.5568  ,  84.085304, 397.01233 , 299.70135 ,   0.999044]],
      dtype=float32)

In [ ]:
for b in dets:
    if b[4] < confience_threshold:
        continue
    text = "{:.2f}".format(b[4])
    b = list(map(int, b))
    cv2.rectangle(img_raw, (b[0], b[1]),(b[2],b[3]),(0,255,0), 2)
    cx, cy = b[0], b[1] + 12
    cv2.putText(img_raw, text, (cx,cy), cv2.FONT_HERSHEY_DUPLEX,0.5,(255,255,255))
    cv2.imshow('RetinaFace Pytorch Detection', img_raw)
    cv2.waitKey(0)
    cv2.destroyAllWindows()